In [1]:
def create_windows(X, Y, window_size = 700 * 60):
    X_windows = []
    Y_windows = []
    
    for i in range(0, len(X) - window_size + 1, window_size):
        X_windows.append(X[i:i+window_size])
        Y_windows.append(mode(Y[i:i+window_size], keepdims=False)[0])
    return np.array(X_windows), np.array(Y_windows)

In [2]:
def test_train_split(X_windows, Y_windows):
    X_train, X_test, y_train, y_test = train_test_split(X_windows, Y_windows, test_size=0.2, random_state=42)
    return X_train, y_train, X_test, y_test

In [3]:
def clean(X_train, X_test, Y_train):

    X_train = X_train.reshape(X_train.shape[0], -1)
    X_test = X_test.reshape(X_test.shape[0], -1)

    smote = SMOTE(random_state=42)
    X_train, Y_train = smote.fit_resample(X_train, Y_train)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, Y_train, X_test

In [4]:
def decision_tree(X_train, X_test, Y_train, Y_test):
    dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')

    dt.fit(X_train, Y_train)
    dt_pred = dt.predict(X_test)
    print("Decision Tree Classification Report:")
    print(classification_report(Y_test, dt_pred, target_names=['baseline', 'stress']))

In [ ]:
import pickle
import numpy as np
from scipy.stats import mode
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import gc


file_paths = [
    'S2.pkl'
]


for file in file_paths:
    with open(file, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    
    chest = data['signal']['chest']
    
    c_ecg = chest['ECG']
    c_emg = chest['EMG']
    c_resp = chest['Resp']
    c_eda = chest['EDA']
    c_temp = chest['Temp']
    c_acc = chest['ACC']
    
    X = np.concatenate([c_acc, c_ecg, c_eda, c_emg, c_resp, c_temp], axis=1)
    
    labels = data['label']
    Y = np.where(labels == 2, 1, 0)

    del data, chest
    gc.collect()

    X_windows,Y_windows = create_windows(X,Y)

    del X, Y
    gc.collect()
    
    X_train, Y_train, X_test, Y_test = test_train_split(X_windows, Y_windows)

    del X_windows, Y_windows

    X_train, Y_train, X_test = clean(X_train, X_test, Y_train)
    decision_tree(X_train, X_test, Y_train, Y_test)